In [1]:
# -*- coding: utf-8 -*-
"""
A股价格联动性分析系统 (2025增强版)
功能：实时监测板块/概念/产业链股票价格联动效应 
环境要求：Python 3.9+ (推荐Anaconda环境)
"""
import akshare as ak 
import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot  as plt
from datetime import datetime, timedelta
from sklearn.covariance  import GraphicalLasso 
from scipy.cluster.hierarchy  import linkage, dendrogram
from ydata_profiling import ProfileReport 

/home/ts/.local/share/virtualenvs/jupyter.13-jNpHegMS/lib/python3.13/site-packages/py_mini_racer/py_mini_racer.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# 配置参数
TODAY = datetime(2025, 7, 30)  # 当前日期
N_DAYS = 120                   # 分析周期 
TOP_CONCEPTS = 10              # 核心联动概念数量 
THRESHOLD = 0.75               # 强联动阈值

In [ ]:
def fetch_market_data():
    """获取全市场股票数据 (2025年实时接口)"""
    print(f"⌛ 正在获取{A股}全市场数据 ({TODAY.strftime('%Y-%m-%d')})...") 
    
    # 获取全量股票代码 
    stock_list = ak.stock_info_a_code_name().set_index('code') 
    
    # 获取历史价格 (优化内存处理)
    price_data = {}
    for i, code in enumerate(stock_list.index[:1000]):   # 演示取前1000只 
        try:
            df = ak.stock_zh_a_hist(symbol=code,  period="daily", 
                                    start_date=(TODAY-timedelta(N_DAYS)).strftime("%Y%m%d"),
                                    end_date=TODAY.strftime("%Y%m%d")) 
            price_data[code] = df.set_index(' 日期')['收盘']
        except:
            continue 
            
        # 进度显示 
        if (i+1) % 100 == 0:
            print(f"▋ 已加载 {i+1}/{len(stock_list)} 只股票数据")
    
    return pd.DataFrame(price_data).astype(float).dropna(axis=1)
 
def compute_linkage_matrix(price_df):
    """计算股票联动矩阵 (基于动态条件相关性)"""
    print("🧮 正在计算价格联动矩阵...")
    
    # 计算对数收益率
    returns = np.log(price_df  / price_df.shift(1)).dropna() 
    
    # 使用图模型估计稀疏协方差矩阵 
    model = GraphicalLasso(alpha=0.05, max_iter=100)
    model.fit(returns) 
    corr_matrix = pd.DataFrame(model.covariance_,  
                               index=returns.columns,  
                               columns=returns.columns) 
    
    # 层次聚类分析 
    linkage_matrix = linkage(corr_matrix, method='ward')
    
    return corr_matrix, linkage_matrix 
 
def visualize_linkage(linkage_matrix, labels):
    """可视化股票联动聚类"""
    plt.figure(figsize=(20,  15))
    dendrogram(linkage_matrix, labels=labels, 
               orientation='right', leaf_font_size=8)
    plt.title(f"A 股市场板块联动聚类分析 ({TODAY.strftime('%Y/%m/%d')})",  fontsize=16)
    plt.axvline(x=THRESHOLD,  color='r', linestyle='--')
    plt.savefig('stock_linkage_dendrogram.png',  dpi=300, bbox_inches='tight')
    print("✅ 联动聚类图已保存至 stock_linkage_dendrogram.png") 
 
def detect_hot_concepts(corr_matrix, stock_data):
    """识别热点联动概念"""
    print("🔍 检测市场热点联动概念...")
    
    # 获取股票概念分类（2025年实时概念库）
    concept_df = ak.stock_board_concept_name_em() 
    
    # 构建概念联动强度矩阵 
    concept_strength = {}
    for concept in concept_df['概念名称'].unique():
        concept_stocks = concept_df[concept_df['概念名称']==concept]['代码']
        valid_stocks = list(set(concept_stocks) & set(corr_matrix.columns)) 
        
        if len(valid_stocks) > 5:  # 忽略成分股过少的概念
            sub_corr = corr_matrix.loc[valid_stocks,  valid_stocks]
            concept_strength[concept] = sub_corr.values.mean() 
    
    # 获取TOP联动概念
    top_concepts = pd.Series(concept_strength).sort_values(ascending=False)[:TOP_CONCEPTS]
    
    # 可视化概念联动强度 
    plt.figure(figsize=(12,  8))
    top_concepts.plot(kind='barh',  color=sns.color_palette("viridis",  len(top_concepts)))
    plt.title(f"TOP{TOP_CONCEPTS} 热点概念联动强度", fontsize=14)
    plt.xlabel(" 平均相关系数")
    plt.tight_layout() 
    plt.savefig('concept_linkage_strength.png',  dpi=150)
    
    return top_concepts
 
def generate_interactive_report(price_df, correlation_matrix):
    """生成交互式分析报告 (2025年增强版)"""
    print("📊 生成交互式分析报告...")
    
    # 创建Profiling报告 
    profile = ProfileReport(price_df, 
                            title="A股价格联动分析报告",
                            correlations={"pearson": {"calculate": False},
                                          "spearman": {"calculate": False},
                                          "kendall": {"calculate": False},
                                          "phi_k": {"calculate": False}})
    
    # 添加自定义关联分析 
    profile.config.correlations  = {
        "custom": {
            "calculate": True,
            "matrix": correlation_matrix
        }
    }
    
    profile.to_file("stock_linkage_report.html") 
    print("✅ 交互式报告已保存至 stock_linkage_report.html") 

In [ ]:
if __name__ == "__main__":
    # 执行主流程 
    print(f"🚀 A股价格联动性分析启动 [日期: {TODAY.strftime('%Y-%m-%d')}]") 
    
    # 数据获取
    price_data = fetch_market_data()
    
    # 联动分析 
    corr_matrix, linkage_mat = compute_linkage_matrix(price_data)
    
    # 可视化 
    visualize_linkage(linkage_mat, price_data.columns) 
    top_concepts = detect_hot_concepts(corr_matrix, price_data)
    
    # 生成报告
    generate_interactive_report(price_data, corr_matrix)
    
    print("\n💎 分析结果摘要:")
    print(f"- 覆盖股票: {len(price_data.columns)}  只")
    print(f"- 最大联动板块: '{top_concepts.index[0]}'  (相关系数: {top_concepts[0]:.3f})")
    print(f"- 强联动股票对: {len(corr_matrix[corr_matrix>THRESHOLD])//2} 组")
    print(f"- 跨概念联动热点: {top_concepts.index[:3].tolist()}") 
    print("\n🎯 操作建议: 关注高联动板块内滞涨股的补涨机会")

In [ ]:
# 采用图模型估算稀疏协方差
model = GraphicalLasso(alpha=0.05, max_iter=100)
model.fit(returns) 
 
# 层次聚类算法 
linkage_matrix = linkage(corr_matrix, method='ward')

In [ ]:
# 识别高联动概念板块 
concept_strength[concept] = sub_corr.values.mean() 
top_concepts = pd.Series(concept_strength).sort_values(ascending=False)[:10]

In [ ]:
# 基于波动率自适应调整阈值 
volatility = returns.std() 
dynamic_threshold = THRESHOLD * (1 + 0.5*(volatility - 0.02)) 

In [ ]:
# 供应链深度分析 (2025年新增)
supply_chain_corr = ak.get_supply_chain_correlation(stock_code) 

In [ ]:
# 监管政策敏感性分析
policy_impact = ak.policy_sensitivity_analysis(stock_code) 